In [ ]:
###--- Load libraries and set the location path for analysis data ---###

In [1]:
# Environment setup
import numpy as np
import scanpy as sc
import pandas as pd
import scipy.io
import matplotlib as mpl
import batchglm.api as glm
import diffxpy.api as de
import decoupler as dc

from matplotlib import rcParams
import bbknn
import os
import sys
import scipy
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scrublet as scr
import scipy.stats as stats

In [2]:
sc.settings.verbosity = 2  # show logging output
sc.settings.dir = "sc_2023_Glyco/"
sc.settings.autosave = True  # save figures, do not show them
sc.settings.figdir = "sc_2023_Glyco/figure"
sc.settings.set_figure_params(dpi=800, format="pdf", dpi_save=800)  # set sufficiently high resolution for saving 400dpi

In [ ]:
###--- Load pre-filtering data ---###

In [3]:
# Data Loading
adata = sc.read("sc_allcells_annotation_global.h5ad")
adata_celltypist_ly = sc.read("sc_lymphoid_clustering.h5ad")
adata_celltypist_my = sc.read("sc_myeloid_clustering.h5ad")

In [4]:
# Create sub_cell_type_spec annotationd
adata_celltypist_ly.obs['comb'] = 'SubLymphoid_' + adata_celltypist_ly.obs['leiden'].astype(str)
adata_celltypist_my.obs['comb'] = 'SubMyeloid_' + adata_celltypist_my.obs['leiden'].astype(str)

# Find the indices of observations that satisfy the condition
selected_indices_my = adata.obs['bc_wells'].isin(adata_celltypist_my.obs['bc_wells'])
selected_indices_ly = adata.obs['bc_wells'].isin(adata_celltypist_ly.obs['bc_wells'])

# Assign values based on selected indices
adata.obs.loc[selected_indices_my, 'cell_type_spec'] = adata_celltypist_my.obs['comb']
adata.obs.loc[selected_indices_ly, 'cell_type_spec'] = adata_celltypist_ly.obs['comb']

# Assign values based on original annotation
adata.obs['cell_type_spec'] = adata.obs['cell_type_spec'].fillna(adata.obs['cell_type'])

del adata_celltypist_ly
del adata_celltypist_my
np.unique(adata.obs['cell_type_spec'], return_counts=True)

(array(['Cancer Cell', 'SubLymphoid_0', 'SubLymphoid_1', 'SubLymphoid_10',
        'SubLymphoid_11', 'SubLymphoid_12', 'SubLymphoid_13',
        'SubLymphoid_14', 'SubLymphoid_15', 'SubLymphoid_2',
        'SubLymphoid_3', 'SubLymphoid_4', 'SubLymphoid_5', 'SubLymphoid_6',
        'SubLymphoid_7', 'SubLymphoid_8', 'SubLymphoid_9', 'SubMyeloid_0',
        'SubMyeloid_1', 'SubMyeloid_10', 'SubMyeloid_11', 'SubMyeloid_12',
        'SubMyeloid_13', 'SubMyeloid_14', 'SubMyeloid_2', 'SubMyeloid_3',
        'SubMyeloid_4', 'SubMyeloid_5', 'SubMyeloid_6', 'SubMyeloid_7',
        'SubMyeloid_8', 'SubMyeloid_9', 'Unknown'], dtype=object),
 array([ 1236, 13042, 11265,  2161,  1125,   970,   723,   579,   246,
         9099,  6821,  5917,  4997,  4120,  3961,  3032,  2426,  4579,
         4128,   995,   937,   524,   210,    72,  3244,  2529,  2414,
         2163,  2141,  1436,  1148,  1122,  1978]))

In [5]:
###--- Basic Preparation  ---###
del adata.uns["log1p"]
# Start with Raw data
adata.X = adata.layers["counts"].copy()

# List of samples
comparison_id= ['CAR1','CAR2','CAR3','Tr2DG1','Tr2DG2','Tr2DG3','TrTUN1','TrTUN2','TrTUN3']

# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask = adata.obs['sample'].isin(comparison_id)
adata = adata[boolean_mask, :]

In [6]:
# Normalization
sc.pp.normalize_total(adata, target_sum=1e4)
adata.layers["norm10k"] = adata.X

# Logarithmize the data:
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X

/home/balestrieri.chiara/anaconda3/envs/spipe/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:169: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


normalizing counts per cell
    finished (0:00:11)


In [ ]:
###--- Ligand-receptor inference ---###

In [9]:
# import rank method via liana
#Rank_Aggregate method that combines the predictions of multiple ligand-receptor method
from liana.mt import rank_aggregate
import liana as li

In [10]:
### LIANA+
li.method.show_methods()

,Method Name,Magnitude Score,Specificity Score,Reference
0,CellPhoneDB,lr_means,cellphone_pvals,"Efremova, M., Vento-Tormo, M., Teichmann, S.A...."
0,Connectome,expr_prod,scaled_weight,"Raredon, M.S.B., Yang, J., Garritano, J., Wang..."
0,log2FC,None,lr_logfc,"Dimitrov, D., Türei, D., Garrido-Rodriguez, M...."
0,NATMI,expr_prod,spec_weight,"Hou, R., Denisenko, E., Ong, H.T., Ramilowski,..."
0,SingleCellSignalR,lrscore,None,"Cabello-Aguilar, S., Alame, M., Kon-Sun-Tack, ..."
0,CellChat,lr_probs,cellchat_pvals,"Jin, S., Guerrero-Juarez, C.F., Zhang, L., Cha..."
0,Rank_Aggregate,magnitude_rank,specificity_rank,"Dimitrov, D., Türei, D., Garrido-Rodriguez, M...."
0,Geometric Mean,lr_gmeans,gmean_pvals,CellPhoneDBv2's permutation approach applied t...


In [11]:
# Annotation
sample_key = 'sample'
condition_key = 'label'
groupby = 'cell_type_spec'

In [12]:
# Ligand-Receptor Inference by Sample
li.mt.rank_aggregate.by_sample(
    adata,
    groupby=groupby,
    sample_key=condition_key, # sample key by which we which to loop
    expr_prop = 0.1,
    use_raw=False,
    n_perms=100,
    return_all_lrs=False,
    verbose=True, 
    )

Now running: CAR:   0%|          | 0/3 [00:00<?, ?it/s]/home/balestrieri.chiara/anaconda3/envs/spipe/lib/python3.9/site-packages/liana/method/_pipe_utils/_pre.py:265: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


... as `zero_center=True`, sparse input is densified and may lead to large memory consumption


Now running: Tr2DG:  33%|███▎      | 1/3 [04:36<09:13, 277.00s/it]/home/balestrieri.chiara/anaconda3/envs/spipe/lib/python3.9/site-packages/liana/method/_pipe_utils/_pre.py:265: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


... as `zero_center=True`, sparse input is densified and may lead to large memory consumption


Now running: TrTUN:  67%|██████▋   | 2/3 [08:35<04:14, 254.44s/it]/home/balestrieri.chiara/anaconda3/envs/spipe/lib/python3.9/site-packages/liana/method/_pipe_utils/_pre.py:265: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


... as `zero_center=True`, sparse input is densified and may lead to large memory consumption


Now running: TrTUN: 100%|██████████| 3/3 [12:51<00:00, 257.26s/it]


In [13]:
adata.uns["liana_res"].sort_values("magnitude_rank")

,label,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,expr_prod,scaled_weight,lr_logfc,spec_weight,lrscore,lr_probs,cellchat_pvals,specificity_rank,magnitude_rank
112247,Tr2DG,SubMyeloid_3,SubMyeloid_6,VCAN,CD44,3.419932,0.00,11.670026,2.542382,3.297713,0.012239,0.977933,0.306005,0.0,0.002574,2.069939e-14
248186,TrTUN,SubMyeloid_3,SubMyeloid_6,VCAN,CD44,3.336502,0.00,11.087989,2.274100,3.123776,0.011288,0.979371,0.282669,0.0,0.004245,3.491566e-14
112248,Tr2DG,SubMyeloid_3,SubMyeloid_4,VCAN,CD44,3.313518,0.00,10.907912,2.456784,3.168164,0.011440,0.977193,0.292510,0.0,0.003493,1.655937e-13
248187,TrTUN,SubMyeloid_3,SubMyeloid_3,VCAN,CD44,3.263555,0.00,10.570522,2.218538,3.071818,0.010761,0.978883,0.275096,0.0,0.004669,2.793225e-13
0,CAR,SubMyeloid_3,SubMyeloid_7,VCAN,CD44,3.235881,0.00,10.429401,2.303419,3.129171,0.011184,0.978648,0.320937,0.0,0.004230,3.054575e-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194995,Tr2DG,SubLymphoid_12,SubLymphoid_5,ADAM17,NOTCH1,0.577378,0.00,0.176082,0.130655,-0.054271,0.000780,0.844809,0.000000,1.0,1.000000,1.000000e+00
194996,Tr2DG,SubLymphoid_12,SubLymphoid_5,YBX1,NOTCH1,0.400429,0.00,0.112101,0.089056,-0.087341,0.000692,0.812857,0.000000,1.0,1.000000,1.000000e+00
194997,Tr2DG,SubLymphoid_11,SubLymphoid_5,IFNG,IFNGR1_IFNGR2,0.288470,0.00,0.078464,1.360615,0.264604,0.009338,0.784198,0.000000,1.0,0.222896,1.000000e+00
194984,Tr2DG,SubLymphoid_13,SubLymphoid_5,NCAM1,PTPRA,0.743615,0.00,0.528110,0.889314,0.959515,0.009007,0.904100,0.001247,0.0,0.017251,1.000000e+00


In [14]:
adata.uns["liana_res"]['source'].unique()

array(['SubMyeloid_3', 'SubMyeloid_13', 'SubMyeloid_5', 'SubMyeloid_6',
       'SubMyeloid_0', 'SubMyeloid_8', 'SubMyeloid_4', 'SubMyeloid_10',
       'SubMyeloid_14', 'SubMyeloid_1', 'SubLymphoid_1', 'SubMyeloid_7',
       'SubMyeloid_2', 'SubLymphoid_13', 'SubLymphoid_0', 'SubLymphoid_7',
       'SubLymphoid_12', 'SubLymphoid_5', 'SubMyeloid_9',
       'SubLymphoid_15', 'Unknown', 'SubLymphoid_2', 'SubLymphoid_10',
       'SubLymphoid_9', 'SubMyeloid_12', 'SubLymphoid_11',
       'SubLymphoid_14', 'Cancer Cell', 'SubLymphoid_8', 'SubMyeloid_11',
       'SubLymphoid_6', 'SubLymphoid_4', 'SubLymphoid_3'], dtype=object)

In [15]:
adata.uns["liana_res"]['ligand_complex'].unique()

array(['VCAN', 'LTF', 'MRC1', 'APP', 'ADAM10', 'S100A9', 'VIM', 'ADAM23',
       'LYZ', 'HDC', 'MAML2', 'CALM1', 'CD55', 'ANXA1', 'S100A8', 'CD22',
       'PSAP', 'HLA-DRB1', 'CD38', 'HSP90B1', 'B2M', 'HLA-DRA', 'COPA',
       'PSEN1', 'HLA-DPB1', 'HLA-DPA1', 'PKM', 'CEACAM8', 'CEACAM1',
       'TIMP2', 'LGALS9', 'CALM2', 'ADAM28', 'PDCD1LG2', 'HLA-DQB1',
       'HLA-DQA1', 'TGS1', 'HLA-B', 'ST6GAL1', 'TNFSF10', 'HBEGF', 'NTN1',
       'TNFSF13B', 'CD59', 'THBS1', 'HLA-A', 'ADAM9', 'ITGB2', 'ADAM17',
       'GNAI2', 'FAM3C', 'NCAM1', 'ENTPD1', 'SELPLG', 'ALCAM', 'COL6A1',
       'GSTP1', 'COL4A4', 'ADAM15', 'SPN', 'LAMC1', 'PLAU', 'TGFB1',
       'ADAM12', 'IL15RA', 'HSPA1A', 'BST1', 'CALM3', 'IL18', 'LAMC2',
       'CALR', 'HLA-E', 'LGALS8', 'MFNG', 'LMAN1', 'PCSK9', 'HSP90AA1',
       'S100A4', 'CD99', 'CFH', 'HGF', 'PAM', 'SERPINA1', 'ANGPTL1',
       'PTDSS1', 'ANXA2', 'F11R', 'HMGB1', 'IGF2', 'HLA-F', 'TCTN1',
       'SIRPA', 'CD47', 'FARP2', 'VEGFA', 'NECTIN2', 'NID1', 'TNF',
   

In [16]:
adata.uns["liana_res"]['receptor_complex'].unique()

array(['CD44', 'IL1RL1', 'PTPRC', 'ITGA4', 'CD74', 'CD36', 'CADM1',
       'ITGAL', 'LRP1', 'HRH2', 'NOTCH2', 'KCNQ5', 'ADGRE5', 'DYSF',
       'APLP2', 'ITGB1', 'CD4', 'PECAM1', 'TLR2', 'CD247', 'ITGB2',
       'SORT1', 'CEACAM1', 'HAVCR2', 'KLRD1', 'KCNQ1', 'TNFRSF21',
       'KLRC2', 'CR1', 'PDCD2', 'RXRA', 'CD22', 'CD68', 'CD69',
       'TNFRSF10A', 'ADORA2B', 'TSPAN14', 'HLA-DPB1', 'STAB1', 'SLC1A5',
       'CCR6', 'PTPRJ', 'ITGAV', 'CD226', 'SELL', 'TNFRSF10D', 'IL6R',
       'CANX', 'CAV1', 'ITGA3', 'CLEC2D', 'ROBO1', 'NRP1', 'ITGA3_ITGB1',
       'TRAF2', 'SIGLEC1', 'MYLK', 'RGMB', 'PTPRK', 'ST14', 'LPP',
       'ITGA9', 'AXL', 'TFRC', 'EGFR', 'TLR4', 'TLR1', 'ESR1',
       'IL18_IL18R1_IL18RAP', 'KLRC2_KLRD1', 'RHBDF2', 'NOTCH1', 'DAG1',
       'PTPRA', 'MCFD2', 'CD81', 'CCR5', 'KLRC1', 'PILRB', 'ITGAM',
       'DPP4', 'ITGA5', 'LILRB2', 'SCARB1', 'MET', 'ITGAL_ITGB2', 'IGF2R',
       'ITGA9_ITGB1', 'IL18BP', 'TMEM67', 'CD47', 'SIRPA', 'KLRK1',
       'PLXNA4', 'CD96', 'IGF1R'

In [20]:
import plotnine as p9
my_plot = (li.pl.dotplot_by_sample(adata, sample_key=condition_key,
                         colour="magnitude_rank", size ="specificity_rank",
                         target_labels=[ 'SubLymphoid_11'],
                         source_labels=['SubMyeloid_0', 'SubMyeloid_1', 'SubMyeloid_5', 'SubMyeloid_6', 'SubMyeloid_14','SubMyeloid_3','SubMyeloid_8','SubMyeloid_4'],
                         ligand_complex=["LGALS9","LGALS3",	"HLA-DRB5", "HLA-DRA", "HLA-DRB1", "HLA-DQA1",  "CD274", "CALR", "ICAM1", "CD86", "CD80"],
                         receptor_complex=["HAVCR2", "LAG3", "CD80", "ITGAV", "IL2RA","CD28"],
                        
                         inverse_colour=True,
                         inverse_size=True,
                                              
                         #size_range=(0.5, 5),                         
                         ) +
    # rotate facet labels
   p9.theme_bw(base_size=14) + p9.scale_color_cmap('RdYlBu_r')+p9.theme(strip_text=p9.element_text(size=10, colour="black", angle=90), figure_size=(8, 20))
)

my_plot.save('dotplot_chat_liana.pdf')

/home/balestrieri.chiara/anaconda3/envs/spipe/lib/python3.9/site-packages/plotnine/scales/scales.py:50: PlotnineWarning: Scale for 'color' is already present.
Adding another scale for 'color',
which will replace the existing scale.

/home/balestrieri.chiara/anaconda3/envs/spipe/lib/python3.9/site-packages/plotnine/ggplot.py:587: PlotnineWarning: Saving 8 x 20 in image.
/home/balestrieri.chiara/anaconda3/envs/spipe/lib/python3.9/site-packages/plotnine/ggplot.py:588: PlotnineWarning: Filename: dotplot_chat_liana.pdf
Fontsize 0.00 < 1.0 pt not allowed by FreeType. Setting fontsize = 1 pt
